# Streaming Content Analytics

Hey! This notebook walks through my little streaming-analytics project: I took the public **MovieLens** dataset (9,742 movies, 100,836 ratings), poked around it, grouped similar movies together, and built a tiny recommender. Think "Netflix intern project." All the heavy lifting lives in `src/` — this notebook is the tour guide.


In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
from IPython.display import Image, Markdown, display

import utils, eda, clustering, recommender
from utils import ensure_dirs, load_data

ensure_dirs()
movies, ratings = load_data()
print(f"movies: {movies.shape}, ratings: {ratings.shape}")
movies.head(3)

## First: what does this data even look like?

Before any fancy modeling, I always just want the lay of the land — how many ratings, who's rating, and whether the ratings skew happy or grumpy (they skew happy, spoiler).


In [ ]:
eda.summary_stats(movies, ratings)

# Make the charts (saved as PNGs into outputs/charts)
eda.chart_ratings_distribution(ratings)
eda.chart_top_genres(movies, ratings)
eda.chart_ratings_over_years(movies, ratings)
eda.chart_genre_heatmap(movies, ratings)

display(Image("../outputs/charts/ratings_distribution.png", width=750))
display(Image("../outputs/charts/top_genres.png", width=750))

No surprises here — the ratings pile up around 3.5 to 4.0, and film-noir/war dramas rate the highest on average. Genres like comedy and horror sit lower, which honestly tracks with how polarizing they are.

The catalog is also *wildly* skewed toward recent decades — the chart below shows title counts climbing fast while the average rating stays pretty flat.


In [ ]:
display(Image("../outputs/charts/ratings_over_years.png", width=800))
display(Image("../outputs/charts/genre_heatmap.png", width=800))

## Next: can we group similar movies together?

I turned each movie into a feature vector — one-hot genres plus normalized release year and average rating — and ran k-means. I let the data pick *k* using silhouette scores instead of just guessing.


In [ ]:
stats, X, _ = clustering.build_features(movies, ratings)
k = clustering.choose_k(X)

from sklearn.cluster import KMeans
labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)

profiles = clustering.cluster_profiles(stats, labels)
display(Markdown(profiles.to_markdown(index=False)))
profiles.to_csv("../outputs/cluster_profiles.csv", index=False)
clustering.scatter_plot(stats, X, labels)
display(Image("../outputs/charts/cluster_scatter.png", width=750))

The clusters actually make sense — you get things like classic dramas hanging out together and kids' movies in their own corner. Not bad for an unsupervised model that never saw a single plot summary.


In [ ]:
# Last stop: a content-based recommender.
# TF-IDF over genres, cosine similarity, popularity breaks ties — simple, but it works.
for title in recommender.SAMPLE_TITLES:
    display(Markdown(f"### If you liked *{title}*..."))
    recs = recommender.recommend(title, n=5)
    display(recs[["title", "genres", "similarity"]])

## What I learned

- **Exploration first:** simple charts answer most questions before any model does.
- **Clustering is a vibe check:** silhouette scores kept me honest about *k*.
- **Small data, real skills:** this whole thing runs offline on a laptop — pandas, scikit-learn, Plotly, DuckDB, and a bit of SQL.

Thanks for reading! The runnable version of all of this is `src/run_all.py`.
